In [1]:
## java runtime is no more available in colab. We'll have to install it manually.
!apt-get install openjdk-21-jdk-headless -qq > /dev/null

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
from pyspark.sql import SparkSession
spark=SparkSession.builder.appName('lung_cancer').getOrCreate()

In [4]:
df=spark.read.csv('survey lung cancer.csv',header=True,inferSchema=True)
df.printSchema()

root
 |-- GENDER: string (nullable = true)
 |-- AGE: integer (nullable = true)
 |-- SMOKING: integer (nullable = true)
 |-- YELLOW_FINGERS: integer (nullable = true)
 |-- ANXIETY: integer (nullable = true)
 |-- PEER_PRESSURE: integer (nullable = true)
 |-- CHRONIC DISEASE: integer (nullable = true)
 |-- FATIGUE : integer (nullable = true)
 |-- ALLERGY : integer (nullable = true)
 |-- WHEEZING: integer (nullable = true)
 |-- ALCOHOL CONSUMING: integer (nullable = true)
 |-- COUGHING: integer (nullable = true)
 |-- SHORTNESS OF BREATH: integer (nullable = true)
 |-- SWALLOWING DIFFICULTY: integer (nullable = true)
 |-- CHEST PAIN: integer (nullable = true)
 |-- LUNG_CANCER: string (nullable = true)



In [5]:
num_col=[i[0] for i in df.dtypes if i[1]=="int"]
cat_col=[i[0] for i in df.dtypes if i[1]!="int"]


In [6]:
from pyspark.ml.feature import VectorAssembler,StringIndexer,OneHotEncoder
stages=[]
for i in cat_col:
  string=StringIndexer(inputCol=i,outputCol=i+'Index')#An encoder defaults frequency encoding
  encoder=OneHotEncoder(inputCols=[string.getOutputCol()],outputCols=[i+"classVec"]) #Converting encoder column to OneHotEncoder
  stages+=[string,encoder]


In [7]:
target=StringIndexer(inputCol='LUNG_CANCER',outputCol='label') #Encoding Target Column
stages+=[target]


In [8]:
assembler=[c+"classVec" for c in cat_col]+num_col


In [9]:
vector_assembler=VectorAssembler(inputCols=assembler,outputCol="features")
stages+=[vector_assembler]


In [17]:
stages

[StringIndexer_1ff367282c01,
 OneHotEncoder_035663239d1a,
 StringIndexer_b5769d337547,
 OneHotEncoder_0cfe1c19d77f,
 StringIndexer_8bf05d06acd3,
 VectorAssembler_9b145650752c]

In [10]:
from pyspark.ml import Pipeline
pipeline=Pipeline(stages=stages)
pipelinemodel=pipeline.fit(df)
df_1=pipelinemodel.transform(df)
df_1.show()

+------+---+-------+--------------+-------+-------------+---------------+--------+--------+--------+-----------------+--------+-------------------+---------------------+----------+-----------+-----------+--------------+----------------+-------------------+-----+--------------------+
|GENDER|AGE|SMOKING|YELLOW_FINGERS|ANXIETY|PEER_PRESSURE|CHRONIC DISEASE|FATIGUE |ALLERGY |WHEEZING|ALCOHOL CONSUMING|COUGHING|SHORTNESS OF BREATH|SWALLOWING DIFFICULTY|CHEST PAIN|LUNG_CANCER|GENDERIndex|GENDERclassVec|LUNG_CANCERIndex|LUNG_CANCERclassVec|label|            features|
+------+---+-------+--------------+-------+-------------+---------------+--------+--------+--------+-----------------+--------+-------------------+---------------------+----------+-----------+-----------+--------------+----------------+-------------------+-----+--------------------+
|     M| 69|      1|             2|      2|            1|              1|       2|       1|       2|                2|       2|                  2| 

In [11]:
select_col=['label','features']
df2=df_1.select(select_col)
df2.show(3)

+-----+--------------------+
|label|            features|
+-----+--------------------+
|  0.0|[1.0,1.0,69.0,1.0...|
|  0.0|[1.0,1.0,74.0,2.0...|
|  1.0|[0.0,0.0,59.0,1.0...|
+-----+--------------------+
only showing top 3 rows


In [12]:
train,test=df2.randomSplit([0.7,0.3])
print("Train Data Count: ",str(train.count()))
print("Test Data Count: ",str(test.count()))

Train Data Count:  218
Test Data Count:  91


In [13]:
from pyspark.ml.classification import LogisticRegression
lr=LogisticRegression(featuresCol='features',labelCol="label",maxIter=10)
lrModel=lr.fit(train)

In [14]:
predictions=lrModel.transform(test)
predictions.show()

+-----+--------------------+--------------------+--------------------+----------+
|label|            features|       rawPrediction|         probability|prediction|
+-----+--------------------+--------------------+--------------------+----------+
|  0.0|[0.0,1.0,47.0,2.0...|[8.67706987592480...|[0.99982957927257...|       0.0|
|  0.0|[0.0,1.0,51.0,2.0...|[7.92479637779742...|[0.99963846726813...|       0.0|
|  0.0|[0.0,1.0,51.0,2.0...|[8.35616360674021...|[0.99976511124129...|       0.0|
|  0.0|[0.0,1.0,51.0,2.0...|[8.35616360674021...|[0.99976511124129...|       0.0|
|  0.0|[0.0,1.0,54.0,2.0...|[9.13091785257764...|[0.99989174555319...|       0.0|
|  0.0|[0.0,1.0,54.0,2.0...|[8.4070264806844,...|[0.99977675701225...|       0.0|
|  0.0|[0.0,1.0,54.0,2.0...|[8.64561877988181...|[0.99982413513321...|       0.0|
|  0.0|[0.0,1.0,54.0,2.0...|[9.02120448422109...|[0.99987919408367...|       0.0|
|  0.0|[0.0,1.0,55.0,1.0...|[8.54525280238406...|[0.99980557172900...|       0.0|
|  0.0|[0.0,1.0,

In [15]:

from  pyspark.ml.evaluation import BinaryClassificationEvaluator
eval=BinaryClassificationEvaluator()
print("Test Area UNder Roc: ",eval.evaluate(predictions))

Test Area UNder Roc:  1.0


In [16]:
from pyspark.ml.classification import DecisionTreeClassifier
dr=DecisionTreeClassifier(featuresCol='features',labelCol="label",maxDepth=3)
drModel=dr.fit(train)
predictions_1=drModel.transform(test)
print("Test Area UNder Roc: ",eval.evaluate(predictions_1))

Test Area UNder Roc:  1.0
